In [ ]:
import os
import json
import pickle
import sympy as sp
import pandas as pd
from IPython.display import display, HTML


In [ ]:
with open('../scripts/configs.json', 'r', encoding='utf-8') as f:
    CONFIGS = json.load(f)
MODELSDIR = CONFIGS['filepaths']['models']
SRCONFIG  = CONFIGS['experiments']['sr']

VARDICT  = {
    'rh':         r'\widehat{\mathrm{RH}}',
    'thetae':     r'\widehat{\theta}_{e}',
    'thetaestar': r'{\widehat{\theta}_{e}^{*}}',
    'bl':         r'\widetilde{\mathrm{B_L}}',
    'lf':         r'\widetilde{\mathrm{LF}}',
    'shf':        r'\widetilde{\mathrm{SHF}}',
    'lhf':        r'\widetilde{\mathrm{LHF}}'}
SYMBOLS  = {k: sp.Symbol(k) for k in VARDICT}
FUNCDICT = {
    'min': sp.Min, 'max': sp.Max, 'abs': sp.Abs,
    'neg': lambda x: -x, 'square': lambda x: x**2,
    'cube': lambda x: x**3, 'exp': sp.exp}

TERMORDER = {'bl': 0, 'rh': 1, 'thetae': 2, 'thetaestar': 3, 'lf': 4, 'shf': 5, 'lhf': 6}

def term_key(term):
    syms = term.free_symbols
    if not syms: return (99, str(term))
    return (min(TERMORDER.get(s.name, 50) for s in syms), str(term))

def order_expr(expr):
    if expr.args:
        expr = expr.func(*[order_expr(a) for a in expr.args], evaluate=False)
    if isinstance(expr, sp.Add):
        expr = sp.Add(*sorted(sp.Add.make_args(expr), key=term_key), evaluate=False)
    return expr

def to_latex(expr):
    latex = sp.latex(expr, symbol_names={SYMBOLS[k]: v for k, v in VARDICT.items()}, mul_symbol='dot')
    return ' '.join(latex.replace(r'\left', '').replace(r'\right', '').split())

CONST_MAP   = {
    'sr_lo':  {'a': 'alpha', 'b': 'beta'},
    'sr_bl':  {'a': 'beta1', 'b': 'beta2'},
    'sr_med': {'a': 'alpha', 'b': 'gamma', 'c': 'eta'},
    'sr_hi':  {'a': 'alpha', 'b': 'gamma', 'c': 'eta', 'd': 'delta'},
}
GREEK_LABEL = {
    'alpha': '\u03b1', 'beta':  '\u03b2',
    'beta1': '\u03b2<sub>1</sub>', 'beta2': '\u03b2<sub>2</sub>',
    'gamma': '\u03b3', 'eta':   '\u03b7', 'delta': '\u03b4',
}
GREEK_SYMS  = {
    'alpha': sp.Symbol('alpha'),  'beta':  sp.Symbol('beta'),
    'beta1': sp.Symbol('beta_1'), 'beta2': sp.Symbol('beta_2'),
    'gamma': sp.Symbol('gamma'),  'eta':   sp.Symbol('eta'),
    'delta': sp.Symbol('delta'),
}
MODEL_ORDER = ['sr_lo', 'sr_bl', 'sr_med', 'sr_hi']

_regpath = os.path.join(MODELSDIR, 'sr', 'optimized_equations.pkl')
registry = pickle.load(open(_regpath, 'rb')) if os.path.exists(_regpath) else {}


In [ ]:
# ── Table 1: optimized constants (inline per model) ──────────────────────────
rows1 = []
for name in MODEL_ORDER:
    eqspec = SRCONFIG['optimizedeqs'].get(name)
    if eqspec is None: continue
    opt   = registry.get(name)
    parts = [f"{GREEK_LABEL[greek]} = {opt['constants'][k]:.4f}"
             for k, greek in CONST_MAP[name].items()
             if opt and k in opt['constants']]
    rows1.append({'Equation': eqspec['description'],
                  'Constants': ', '.join(parts) or '\u2013'})
display(HTML(pd.DataFrame(rows1).set_index('Equation').to_html(escape=False)))

# ── Table 2: symbolic equation forms (left-aligned) ──────────────────────────
rows2 = []
for name in MODEL_ORDER:
    eqspec = SRCONFIG['optimizedeqs'].get(name)
    if eqspec is None: continue
    opt = registry.get(name)
    if opt:
        ns = {**SYMBOLS, **FUNCDICT, **{k: GREEK_SYMS[v] for k, v in CONST_MAP[name].items()}}
        try:
            form = '$' + to_latex(order_expr(eval(opt['form'], {'__builtins__': {}}, ns))) + '$'
        except Exception as e:
            form = f"{opt['form']}  [error: {e}]"
    else:
        form = '\u2013'
    rows2.append({'Equation': eqspec['description'], 'Form': form})

_css = '<style>.eq-tbl td { text-align: left; }</style>'
display(HTML(_css + pd.DataFrame(rows2).set_index('Equation').to_html(escape=False, classes='eq-tbl')))
